# Partie 1 : Exploration des Transformers et tâches NLP


Ce notebook couvre :
1. Installation & imports
2. Chargement de modèles pré-entraînés et tokenizers
3. Classification de texte
4. Analyse de sentiment
5. Question Answering
6. Résumé automatique


## 0. Installation des dépendances

In [1]:
import subprocess, sys

pkgs = ["transformers", "torch", "datasets",
        "sentencepiece", "sacremoses", "accelerate"]

for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip",
                           "install", "--quiet", pkg])

print("✅ Installation terminée — redémarrez le kernel puis réexécutez")

✅ Installation terminée — redémarrez le kernel puis réexécutez


In [2]:
import transformers, torch
print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}")
print("✅ Tout est prêt !")

c:\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers : 5.9.0
torch        : 2.7.0+cpu
✅ Tout est prêt !


## 1. Imports et configuration générale

In [3]:
# ─────────────────────────────────────────────
# Imports principaux
# ─────────────────────────────────────────────
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
)
import torch
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports réussis")
print(f"🔥 PyTorch version : {torch.__version__}")
print(f"🤗 Transformers prêts")

✅ Imports réussis
🔥 PyTorch version : 2.7.0+cpu
🤗 Transformers prêts


---
## 2. Tokenization — Comprendre comment les Transformers lisent le texte

Le **tokenizer** est la première étape de tout pipeline NLP. Il convertit le texte brut en séquence d'identifiants (tokens IDs) que le modèle peut traiter.

Étapes principales :
- **Segmentation** : découpage du texte en sous-mots (WordPiece, BPE, SentencePiece)
- **Encodage** : mapping token → entier
- **Padding/Truncation** : normalisation de la longueur des séquences
- **Attention mask** : indiquer au modèle quels tokens sont réels vs padding

In [4]:
# ─────────────────────────────────────────────
# 2.1  Chargement d'un tokenizer pré-entraîné
# ─────────────────────────────────────────────
model_checkpoint = "bert-base-uncased"  # BERT de Google
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Modèle       : {model_checkpoint}")
print(f"Vocabulaire  : {tokenizer.vocab_size} tokens")
print(f"Type         : {tokenizer.__class__.__name__}")

Modèle       : bert-base-uncased
Vocabulaire  : 30522 tokens
Type         : BertTokenizer


In [5]:
# ─────────────────────────────────────────────
# 2.2  Tokenisation d'une phrase
# ─────────────────────────────────────────────
texte = "Transformers have revolutionized natural language processing."

# Tokenisation simple → liste de tokens sous forme de chaînes
tokens = tokenizer.tokenize(texte)
print("Tokens :", tokens)

# Encodage complet → dictionnaire avec input_ids, attention_mask
encoded = tokenizer(texte, return_tensors="pt")
print("\nInput IDs    :", encoded["input_ids"])
print("Attention Mask:", encoded["attention_mask"])

# Décodage : repasser de IDs → texte
decoded = tokenizer.decode(encoded["input_ids"][0])
print("\nTexte décodé :", decoded)

Tokens : ['transformers', 'have', 'revolution', '##ized', 'natural', 'language', 'processing', '.']

Input IDs    : tensor([[  101, 19081,  2031,  4329,  3550,  3019,  2653,  6364,  1012,   102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Texte décodé : [CLS] transformers have revolutionized natural language processing. [SEP]


In [6]:
# ─────────────────────────────────────────────
# 2.3  Batch de phrases avec padding automatique
# ─────────────────────────────────────────────
phrases = [
    "BERT is a powerful language model.",
    "HuggingFace makes transformers easy to use.",
    "NLP is fun!"
]

batch = tokenizer(
    phrases,
    padding=True,       # Pad les séquences courtes
    truncation=True,    # Tronque les séquences trop longues
    max_length=32,      # Longueur max
    return_tensors="pt"
)

print("Shape du batch input_ids :", batch["input_ids"].shape)
print("\nInput IDs du batch :")
print(batch["input_ids"])
print("\nAttention masks :")
print(batch["attention_mask"])

Shape du batch input_ids : torch.Size([3, 10])

Input IDs du batch :
tensor([[  101, 14324,  2003,  1037,  3928,  2653,  2944,  1012,   102,     0],
        [  101, 17662, 12172,  3084, 19081,  3733,  2000,  2224,  1012,   102],
        [  101, 17953,  2361,  2003,  4569,   999,   102,     0,     0,     0]])

Attention masks :
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


---
## 3. Classification de texte (Text Classification)

La classification de texte consiste à assigner une **catégorie** à un texte donné.  
Exemples d'applications : détection de spam, classification de sujets, modération de contenu.

Nous utilisons un modèle fine-tuné sur le dataset AG News (4 catégories : World, Sports, Business, Sci/Tech).

In [7]:
# ─────────────────────────────────────────────
# 3.1  Via le Pipeline simplifié (méthode rapide)
# ─────────────────────────────────────────────
classifier = pipeline(
    task="text-classification",
    model="fabriceyhc/bert-base-uncased-ag_news",  # Fine-tuné sur AG News
)

textes_test = [
    "NASA successfully launches a new rocket to the Moon.",
    "The stock market crashed today due to inflation fears.",
    "Barcelona wins the Champions League final against Real Madrid.",
    "Scientists discover a new cure for Alzheimer's disease."
]

results = classifier(textes_test)

print("=" * 65)
print(f"{'Texte':<50} {'Classe':<12} {'Score'}")
print("=" * 65)
for texte, res in zip(textes_test, results):
    print(f"{texte[:48]:<50} {res['label']:<12} {res['score']:.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32229.34it/s]


Texte                                              Classe       Score
NASA successfully launches a new rocket to the M   LABEL_3      0.9855
The stock market crashed today due to inflation    LABEL_2      0.9962
Barcelona wins the Champions League final agains   LABEL_1      0.6311
Scientists discover a new cure for Alzheimer's d   LABEL_0      0.9877


In [8]:
# ─────────────────────────────────────────────
# 3.2  Via AutoTokenizer + AutoModel (méthode détaillée)
#      Utile pour comprendre le fonctionnement interne
# ─────────────────────────────────────────────
import torch
import torch.nn.functional as F

model_name = "fabriceyhc/bert-base-uncased-ag_news"
tokenizer_cls = AutoTokenizer.from_pretrained(model_name)
model_cls = AutoModelForSequenceClassification.from_pretrained(model_name)
model_cls.eval()

texte_exemple = "The president signs a new trade agreement with Europe."

# Encodage
inputs = tokenizer_cls(texte_exemple, return_tensors="pt", truncation=True, max_length=128)

# Inférence sans calcul de gradient
with torch.no_grad():
    outputs = model_cls(**inputs)

# Logits → probabilités
logits = outputs.logits
probs = F.softmax(logits, dim=-1)

# Affichage des probabilités pour chaque classe
labels = model_cls.config.id2label
print(f"Texte : {texte_exemple}\n")
print("Probabilités par classe :")
for idx, prob in enumerate(probs[0]):
    print(f"  {labels[idx]:<12} : {prob.item():.4f}")

pred_label = labels[probs.argmax().item()]
print(f"\n✅ Classe prédite : {pred_label}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28869.77it/s]

Texte : The president signs a new trade agreement with Europe.

Probabilités par classe :
  LABEL_0      : 0.9329
  LABEL_1      : 0.0007
  LABEL_2      : 0.0576
  LABEL_3      : 0.0088

✅ Classe prédite : LABEL_0


---
## 4. Analyse de Sentiment (Sentiment Analysis)

L'analyse de sentiment détermine la **polarité émotionnelle** d'un texte : positif, négatif, ou neutre.  
Applications : avis clients, monitoring des réseaux sociaux, feedback produit.

In [9]:
# ─────────────────────────────────────────────
# 4.1  Sentiment binaire (POSITIVE / NEGATIVE)
# ─────────────────────────────────────────────
# distilbert fine-tuné sur SST-2 (Stanford Sentiment Treebank)
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

avis_clients = [
    "This product is absolutely amazing! I love it.",
    "Terrible experience. The quality is very poor and disappointing.",
    "It's okay, nothing special but not bad either.",
    "I'm so happy with my purchase. Highly recommend!",
    "The customer service was rude and unhelpful. Never buying again."
]

print("Analyse de sentiment (SST-2)")
print("=" * 75)
for avis in avis_clients:
    result = sentiment_pipeline(avis)[0]
    emoji = "😊" if result['label'] == 'POSITIVE' else "😠"
    print(f"{emoji} [{result['label']:<8} {result['score']:.3f}] {avis[:60]}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9024.86it/s]


Analyse de sentiment (SST-2)
😊 [POSITIVE 1.000] This product is absolutely amazing! I love it.
😠 [NEGATIVE 1.000] Terrible experience. The quality is very poor and disappoint
😊 [POSITIVE 0.994] It's okay, nothing special but not bad either.
😊 [POSITIVE 1.000] I'm so happy with my purchase. Highly recommend!
😠 [NEGATIVE 0.999] The customer service was rude and unhelpful. Never buying ag


In [14]:
# ─────────────────────────────────────────────
# 4.2  Sentiment multi-classes (3 classes)
# ─────────────────────────────────────────────
sentiment_multi = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    top_k=None        
)


tweet = "I just got the new iPhone. It's pretty good but a bit expensive."
output = sentiment_multi(tweet)

# Selon la version de transformers, le format peut être :
# [[{label, score}, ...]]  ou  [{label, score}, ...]
# On normalise :
if isinstance(output[0], list):
    scores = output[0]   # format [[...]]
else:
    scores = output      # format [...]

label_map = {"LABEL_0": "Négatif 😠", "LABEL_1": "Neutre 😐", "LABEL_2": "Positif 😊"}

print(f"Tweet : {tweet}\n")
print("Scores par sentiment :")
for item in scores:
    label = label_map.get(item['label'], item['label'])
    bar = "█" * int(item['score'] * 30)
    print(f"  {label:<15} : {item['score']:.4f}  {bar}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21628.44it/s]


Tweet : I just got the new iPhone. It's pretty good but a bit expensive.

Scores par sentiment :
  Positif 😊       : 0.9695  █████████████████████████████
  Neutre 😐        : 0.0266  
  Négatif 😠       : 0.0038  


---
## 5. Question Answering (QA)

Le **Question Answering extractif** consiste à extraire la réponse à une question directement depuis un **contexte** fourni.  
Le modèle prédit les positions de début et fin du span de réponse dans le texte.

Modèle utilisé : **DistilBERT fine-tuné sur SQuAD v1.1** (Stanford Question Answering Dataset)

In [15]:
# ─────────────────────────────────────────────
# 5.1  Question Answering (compatible transformers 5.x)
# ─────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name   = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model     = AutoModelForQuestionAnswering.from_pretrained(model_name)
qa_model.eval()

context = """
The Transformer architecture was introduced in the paper 'Attention Is All You Need' 
by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms and 
dispenses with recurrence and convolutions entirely. BERT, developed by Google in 2018, 
is a bidirectional transformer pre-trained using masked language modeling and 
next sentence prediction. GPT (Generative Pre-trained Transformer) was developed by 
OpenAI and is a unidirectional transformer focused on text generation. 
HuggingFace was founded in 2016 and provides the Transformers library which 
democratizes access to pre-trained models.
"""

questions = [
    "Who introduced the Transformer architecture?",
    "When was BERT developed?",
    "What does BERT stand for in terms of directionality?",
    "Who developed GPT?",
    "When was HuggingFace founded?"
]

def repondre(question, context):
    inputs = qa_tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start = outputs.start_logits.argmax()
    end   = outputs.end_logits.argmax() + 1

    # Score = moyenne des logits max début + fin
    score = (outputs.start_logits.max() + outputs.end_logits.max()).item() / 2

    tokens = inputs["input_ids"][0][start:end]
    answer = qa_tokenizer.decode(tokens, skip_special_tokens=True)
    return answer, score, start.item(), end.item()

print("Question Answering sur un passage\n")
print("=" * 70)
for question in questions:
    answer, score, start, end = repondre(question, context)
    print(f"❓ {question}")
    print(f"✅ Réponse : {answer}  (score: {score:.4f})")
    print(f"   Position : [{start} → {end}]")
    print()

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8628.51it/s]


Question Answering sur un passage

❓ Who introduced the Transformer architecture?
✅ Réponse : Vaswani et al.  (score: 11.5796)
   Position : [28 → 35]

❓ When was BERT developed?
✅ Réponse : 2018  (score: 11.1514)
   Position : [69 → 70]

❓ What does BERT stand for in terms of directionality?
✅ Réponse : Generative Pre - trained Transformer  (score: 6.0147)
   Position : [99 → 106]

❓ Who developed GPT?
✅ Réponse : OpenAI  (score: 11.7387)
   Position : [102 → 105]

❓ When was HuggingFace founded?
✅ Réponse : 2016  (score: 11.5960)
   Position : [129 → 130]



In [16]:
# ─────────────────────────────────────────────
# 5.2  QA manuel avec AutoModel (accès aux logits)
# ─────────────────────────────────────────────
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_model.eval()

question = "In what year was the Transformer introduced?"
context_court = "The Transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017."

# Encodage question + contexte
inputs = qa_tokenizer(
    question,
    context_court,
    return_tensors="pt",
    return_offsets_mapping=False,
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = qa_model(**inputs)

# Le modèle retourne des logits de début et de fin
start_logits = outputs.start_logits
end_logits   = outputs.end_logits

start_idx = start_logits.argmax()
end_idx   = end_logits.argmax() + 1

# Décoder les tokens correspondant au span prédit
answer_tokens = inputs["input_ids"][0][start_idx:end_idx]
answer = qa_tokenizer.decode(answer_tokens)

print(f"❓ Question : {question}")
print(f"📄 Contexte : {context_court}")
print(f"\n✅ Réponse extraite : {answer}")
print(f"   Span tokens : [{start_idx.item()} → {end_idx.item()}]")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 9036.01it/s]


❓ Question : In what year was the Transformer introduced?
📄 Contexte : The Transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017.

✅ Réponse extraite : 2017
   Span tokens : [38 → 39]


---
## 6. Résumé Automatique (Text Summarization)

La **summarization** compresse un texte long en gardant l'information essentielle.  
Deux approches existent :
- **Extractive** : sélection des phrases les plus importantes du texte source
- **Abstractive** : génération de nouvelles phrases résumant le contenu (comme un humain)

Nous utilisons **BART** et **T5**, deux modèles seq2seq fine-tunés sur CNN/DailyMail.

In [18]:
# ─────────────────────────────────────────────
# 6.1  Résumé avec BART (compatible transformers 5.x)
# ─────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name    = "sshleifer/distilbart-cnn-12-6"
tokenizer_sum = AutoTokenizer.from_pretrained(model_name)
model_sum     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model_sum.eval()

article = """
Artificial intelligence (AI) is transforming every industry at an unprecedented pace. 
In healthcare, AI algorithms can now diagnose diseases like cancer with accuracy 
comparable to experienced doctors. In finance, machine learning models detect 
fraudulent transactions in real time, saving billions of dollars annually. 
The transportation sector is being disrupted by autonomous vehicles powered by deep 
learning. Companies like Tesla, Waymo, and Cruise are leading the charge in 
developing self-driving cars that could reduce road accidents by up to 90%. 
In education, personalized AI tutors adapt to each student's learning pace and style, 
making quality education accessible to all. Natural Language Processing breakthroughs, 
particularly large language models like GPT-4 and Claude, are enabling machines to 
understand and generate human language with remarkable fluency. Despite these advances, 
experts warn of risks including job displacement, algorithmic bias, and privacy concerns 
that must be carefully managed through regulation and ethical guidelines.
"""

print(f"📄 Texte original ({len(article.split())} mots)\n")

inputs = tokenizer_sum(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

with torch.no_grad():
    summary_ids = model_sum.generate(
        inputs["input_ids"],
        max_new_tokens=120,
        min_new_tokens=30,
        num_beams=4,
        early_stopping=True
    )

summary_bart = tokenizer_sum.decode(summary_ids[0], skip_special_tokens=True)
print(f"📝 Résumé BART ({len(summary_bart.split())} mots) :")
print(summary_bart)
print(f"\n📊 Taux de compression : {len(summary_bart.split())/len(article.split()):.1%}")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 30791.77it/s]


📄 Texte original (147 mots)



[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Résumé BART (40 mots) :
 Artificial Intelligence is transforming every industry at an unprecedented pace . It is being used to diagnose diseases like cancer with accuracy in real-time . In the future, it will be used to make life-saving decisions easier to make .

📊 Taux de compression : 27.2%


In [19]:
# ─────────────────────────────────────────────
# 6.2  Résumé avec T5
# ─────────────────────────────────────────────
model_name_t5  = "t5-small"
tokenizer_t5   = AutoTokenizer.from_pretrained(model_name_t5)
model_t5       = AutoModelForSeq2SeqLM.from_pretrained(model_name_t5)
model_t5.eval()

# T5 nécessite le préfixe "summarize: "
inputs_t5 = tokenizer_t5(
    "summarize: " + article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    summary_ids_t5 = model_t5.generate(
        inputs_t5["input_ids"],
        max_new_tokens=100,
        min_new_tokens=20,
        num_beams=4,
        early_stopping=True
    )

summary_t5 = tokenizer_t5.decode(summary_ids_t5[0], skip_special_tokens=True)
print(f"📝 Résumé T5 ({len(summary_t5.split())} mots) :")
print(summary_t5)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 11015.51it/s]


📝 Résumé T5 (42 mots) :
AI algorithms can diagnose diseases like cancer with accuracy comparable to experienced doctors. in finance, machine learning models detect fraudulent transactions in real time, saving billions of dollars annually. the transportation sector is being disrupted by autonomous vehicles powered by deep learning.


In [20]:
# ─────────────────────────────────────────────
# 6.3  Comparaison des deux résumés
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("COMPARAISON DES RÉSUMÉS")
print("=" * 60)
print(f"\n🔵 BART ({len(summary_bart.split())} mots):")
print(summary_bart)
print(f"\n🟢 T5 ({len(summary_t5.split())} mots):")
print(summary_t5)
print(f"\n📄 Original : {len(article.split())} mots")


COMPARAISON DES RÉSUMÉS

🔵 BART (40 mots):
 Artificial Intelligence is transforming every industry at an unprecedented pace . It is being used to diagnose diseases like cancer with accuracy in real-time . In the future, it will be used to make life-saving decisions easier to make .

🟢 T5 (42 mots):
AI algorithms can diagnose diseases like cancer with accuracy comparable to experienced doctors. in finance, machine learning models detect fraudulent transactions in real time, saving billions of dollars annually. the transportation sector is being disrupted by autonomous vehicles powered by deep learning.

📄 Original : 147 mots


---
## 7. Bonus : Zero-Shot Classification

La **Zero-Shot Classification** permet de classer un texte dans des catégories **non vues pendant l'entraînement**.  
Très utile quand on n'a pas de données labelisées. Le modèle utilise son compréhension du langage naturel pour décider.

In [21]:
# ─────────────────────────────────────────────
# Zero-Shot Classification avec BART-MNLI
# ─────────────────────────────────────────────
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

textes = [
    "The government announced new policies to reduce carbon emissions by 2030.",
    "The new iPhone 16 features a faster chip and improved camera system.",
    "Researchers developed a new vaccine with 95% efficacy against the flu virus."
]

candidate_labels = ["politics", "technology", "health", "sports", "economy"]

print("Zero-Shot Classification\n")
for texte in textes:
    result = zero_shot(texte, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"📝 {texte[:70]}")
    print(f"   → Catégorie : {top_label.upper()} (score: {top_score:.3f})")
    print()

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4470.12it/s]


Zero-Shot Classification

📝 The government announced new policies to reduce carbon emissions by 20
   → Catégorie : POLITICS (score: 0.566)

📝 The new iPhone 16 features a faster chip and improved camera system.
   → Catégorie : TECHNOLOGY (score: 0.949)

📝 Researchers developed a new vaccine with 95% efficacy against the flu 
   → Catégorie : TECHNOLOGY (score: 0.524)



---
## 8. Récapitulatif

| Tâche | Modèle utilisé | Dataset d'entraînement |
|---|---|---|
| Tokenization | BERT-base-uncased | Wikipedia + BookCorpus |
| Classification de texte | BERT fine-tuné | AG News |
| Analyse de sentiment (binaire) | DistilBERT | SST-2 |
| Analyse de sentiment (multi) | RoBERTa | Twitter |
| Question Answering | DistilBERT | SQuAD v1.1 |
| Résumé automatique | DistilBART + T5 | CNN/DailyMail |
| Zero-Shot Classification | BART-large | MNLI |

### Points clés retenus
- `pipeline()` de HuggingFace permet d'utiliser n'importe quelle tâche NLP en **3 lignes de code**
- `AutoTokenizer` + `AutoModel` offre un contrôle fin sur l'encodage et les prédictions
- Les modèles pré-entraînés sont téléchargés depuis le **HuggingFace Hub** et mis en cache localement
- Le choix du modèle (BERT, RoBERTa, T5, BART) influence directement la qualité des résultats

